# LLM Evaluation (LLM-as-a-Judge)

This notebook demonstrates how to use one Large Language Model (LLM) to evaluate the outputs of another. This technique, known as "LLM-as-a-Judge," is becoming increasingly popular for scalable and automated evaluation.

**Task:** Evaluate the quality of generated answers.
**Models:** 
*   **Generator:** `HuggingFaceTB/SmolLM2-1.7B-Instruct`
*   **Judge:** `Qwen/Qwen2.5-1.5B-Instruct`

## License

This notebook is licensed under the Apache License 2.0. See the [LICENSE](https://github.com/huggingface/transformers/blob/main/LICENSE) file for more details.

This notebook focuses on open-source, lightweight, and updated models, and is optimized to run on the Google Colab Free Tier.

## 1. Installation

In [ ]:
!pip install -q transformers torch accelerate bitsandbytes

## 2. Imports

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import json

## 3. Load Data

In [ ]:
# We will use a small set of test questions
questions = [
    "Explain the concept of quantum entanglement to a 5-year-old.",
    "Write a python function to calculate the fibonacci sequence.",
    "What are the health benefits of eating apples?"
]

print(f"Loaded {len(questions)} questions.")

## 4. Data Cleaning

In [ ]:
# Ensure questions are strings and stripped of whitespace
questions = [q.strip() for q in questions if isinstance(q, str)]
print("Data cleaned.")

## 5. Load Model

In [ ]:
# 1. Load the Generator Model (SmolLM2)
generator_model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
print(f"Loading Generator: {generator_model_id}...")
generator_tokenizer = AutoTokenizer.from_pretrained(generator_model_id)
generator_pipe = pipeline(
    "text-generation",
    model=generator_model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)

# 2. Load the Judge Model (Qwen2.5)
judge_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading Judge: {judge_model_id}...")
judge_tokenizer = AutoTokenizer.from_pretrained(judge_model_id)
judge_pipe = pipeline(
    "text-generation",
    model=judge_model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)

## 6. Inference

In [ ]:
# Step 1: Generate Answers
answers = []
print("Generating answers...")
for q in questions:
    messages = [{"role": "user", "content": q}]
    prompt = generator_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = generator_pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.7)
    generated_text = outputs[0]["generated_text"]
    # Extract just the response (simple split for this model/template)
    # Note: Adjust splitting logic based on actual template output if needed
    answer = generated_text.split("assistant\n")[-1].strip()
    answers.append({"question": q, "answer": answer})
    print(f"Generated answer for: {q[:30]}...")

# Step 2: Evaluate with Judge
results = []
print("\nEvaluating answers...")
for item in answers:
    # Construct evaluation prompt
    eval_prompt = f"""You are a strict judge. Evaluate the following answer to the user's question.
Question: {item['question']}
Answer: {item['answer']}

Rate the answer on a scale of 1 to 10 based on accuracy and clarity. Provide a short explanation.
Format: Score: [Score] - Explanation: [Text]"""
    
    messages = [{"role": "user", "content": eval_prompt}]
    prompt = judge_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = judge_pipe(prompt, max_new_tokens=128, do_sample=False)
    evaluation = outputs[0]["generated_text"].split("assistant\n")[-1].strip()
    
    results.append({
        "question": item['question'],
        "answer": item['answer'],
        "evaluation": evaluation
    })
    print(f"Evaluated: {evaluation[:50]}...")

print("\nFinal Results:")
for r in results:
    print(r)

## 7. Save Results

In [ ]:
# Kitchen Sink: Save Results
import os
import numpy as np
try:
    from PIL import Image
except ImportError:
    pass
try:
    import scipy.io.wavfile as wavfile
except ImportError:
    pass

print("Saving results...")

# 1. Text / Data / Variables
with open('results.txt', 'w') as f:
    if 'results' in locals():
        f.write(str(results))
    elif 'predictions' in locals():
        f.write(str(predictions))
    elif 'prediction' in locals():
        f.write(str(prediction))
    elif 'output' in locals():
        f.write(str(output))
    elif 'features' in locals():
        f.write(str(features))
    elif 'summary' in locals():
        f.write(str(summary))
    elif 'translation' in locals():
        f.write(str(translation))
    elif 'answers' in locals():
        f.write(str(answers))
    elif 'conversation' in locals():
        f.write(str(conversation))
    elif 'generated_text' in locals():
        f.write(str(generated_text))
    elif 'transcription' in locals():
        f.write(str(transcription))
    elif 'text' in locals():
        f.write(str(text))
    elif 'triplets' in locals():
        f.write(str(triplets))
    else:
        f.write("Task completed.")
print("Saved results.txt")

# 2. Images
if 'image' in locals():
    if hasattr(image, 'save'):
        image.save('generated_image.png')
        print("Saved generated_image.png")
elif 'images' in locals() and isinstance(images, list) and len(images) > 0:
    if hasattr(images[0], 'save'):
        images[0].save('generated_image.png')
        print("Saved generated_image.png (first image)")
    # Try to save as GIF if it's a list of images (for video/3D)
    try:
        from diffusers.utils import export_to_gif
        export_to_gif(images, "generated_video.gif")
        print("Saved generated_video.gif")
    except:
        # Fallback for standard list of PIL images
        try:
            images[0].save("generated_video.gif", save_all=True, append_images=images[1:], duration=100, loop=0)
            print("Saved generated_video.gif")
        except:
            pass

# Graph Visualization (specific to Graph RAG)
if os.path.exists("graph_visualization.png"):
    import shutil
    shutil.copy("graph_visualization.png", "generated_image.png")
    print("Saved generated_image.png from graph_visualization.png")

# 3. Audio
if 'audio' in locals():
    if isinstance(audio, tuple):
        rate, data = audio
        try:
            wavfile.write('generated_audio.wav', rate, data)
            print("Saved generated_audio.wav")
        except:
            pass
    elif hasattr(audio, 'ndim'): # numpy array
        rate = save_rate if 'save_rate' in locals() else 16000
        try:
            wavfile.write('generated_audio.wav', rate, audio)
            print("Saved generated_audio.wav")
        except:
            pass
elif 'audio_data' in locals():
    rate = save_rate if 'save_rate' in locals() else 16000
    try:
        wavfile.write('generated_audio.wav', rate, audio_data)
        print("Saved generated_audio.wav")
    except:
        pass


## 8. Advanced Usage

**Teaching Tip:** Try varying the input parameters or prompts to see how the model's output changes. This helps build intuition about the model's capabilities and limitations.

Try using different judge models (e.g., larger models via API) or defining more specific rubrics (e.g., "Evaluate only for code correctness").